# Folio

Enable GPU and Internet, then **Run All**.
For the web UI, use the HTTP URL printed by Cloudflare.
Interrupt the cell to stop all three processes.


In [ ]:
%%bash
set -euo pipefail
[ -d folio ] || git clone https://github.com/lzhoang2801/folio.git folio
cd folio
python -m pip install -q uv==0.12.19
uv sync --frozen --no-dev --extra vllm --python 3.12
curl -fsSL --retry 3 https://github.com/cloudflare/cloudflared/releases/download/2026.10.4/cloudflared-linux-amd64 -o .venv/bin/cloudflared
printf '%s  %s\n' ae5e8add7884b0531d66bbca4570d1de491ff6da51e4ebe7eb160cdf241d9e42 .venv/bin/cloudflared | sha256sum -c -
chmod +x .venv/bin/cloudflared

set -m
.venv/bin/folio serve-vllm &
model_pid=$!
trap 'kill -- -"$model_pid" -"${web_pid:-$model_pid}" -"${tunnel_pid:-$model_pid}" 2>/dev/null || true; wait 2>/dev/null || true' EXIT
trap 'exit 130' INT
trap 'exit 143' TERM
.venv/bin/folio web --port 8000 --base-url http://127.0.0.1:8010/v1 &
web_pid=$!
.venv/bin/cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate &
tunnel_pid=$!
wait -n "$model_pid" "$web_pid" "$tunnel_pid"
